# ch03 异常检测与抗干扰分析

**目标**：对比正常与异常工况下传感器信号差异，提取故障敏感特征

**依赖**：ch02

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path('.') / '..'))
from src.utils.data_loader import load_anomaly_data, split_by_label
from src.utils.metrics import perform_ttest, perform_ks_test, extract_time_features
from src.utils.output_manager import save_dataframe

import pandas as pd

CHAPTER_ID = 'ch03'

## 1. 数据加载与分组

In [ ]:
df = load_anomaly_data()
groups = split_by_label(df)
print(f"Label=0 (正常): {len(groups[0])} 行")
print(f"Label=1 (卡滞): {len(groups[1])} 行")
print(f"Label=2 (脱扣): {len(groups[2])} 行")

## 2. 描述性统计对比

In [ ]:
key_signals = ['MotorData.ActCurrent', 'MotorData.ActPosition', 'MotorData.ActSpeed']
comparison_records = []

for signal in key_signals:
    if signal in df.columns:
        for label, group_df in groups.items():
            comparison_records.append({
                'signal': signal,
                'label': label,
                'mean': group_df[signal].mean(),
                'std': group_df[signal].std(),
                'count': len(group_df),
            })

comparison_df = pd.DataFrame(comparison_records)
comparison_df

## 3. 统计检验

In [ ]:
test_results = []
normal_group = groups[0]

for signal in key_signals:
    if signal in df.columns:
        for anomaly_label in [1, 2]:
            ttest = perform_ttest(normal_group[signal], groups[anomaly_label][signal])
            kstest = perform_ks_test(normal_group[signal], groups[anomaly_label][signal])
            test_results.append({
                'signal': signal,
                'anomaly_label': anomaly_label,
                't_pvalue': ttest['p_value'],
                't_significant': ttest['significant'],
                'ks_pvalue': kstest['p_value'],
                'ks_significant': kstest['significant'],
                'cohens_d': ttest['cohens_d'],
            })

test_df = pd.DataFrame(test_results)
test_df

## 4. 特征提取

In [ ]:
feature_records = []
for signal in key_signals:
    if signal in df.columns:
        for label, group_df in groups.items():
            features = extract_time_features(group_df[signal])
            features['signal'] = signal
            features['label'] = label
            feature_records.append(features)

feature_df = pd.DataFrame(feature_records)
feature_df

## 5. 保存产物

In [ ]:
save_dataframe(comparison_df, 'normal_vs_anomaly_stats.csv', CHAPTER_ID)
save_dataframe(test_df, 'statistical_test_results.csv', CHAPTER_ID)
save_dataframe(feature_df, 'distortion_features_table.csv', CHAPTER_ID)